# Sentinel — Adaptive Time-Series Anomaly Detection

Compact end-to-end notebook using the NAB `ambient_temperature_system_failure.csv` dataset.

- Loads the CSV from `/content`, prompts for upload in Colab, or downloads the public NAB copy if needed.
- Cleans the data and computes a causal rolling-median baseline with robust MAD scaling.
- Selects the best threshold by point-level F1 across the tested thresholds.
- Reports precision, recall, F1, anomaly counts, plots detections, and exports predictions.

**Previously recorded run from your original notebook:** threshold **1.0**, precision **0.109**, recall **0.471**, F1 **0.178**, with 3,125 flagged readings, 4,092 normal readings, and 50 warm-up readings. The code below recalculates these results from the dataset at runtime rather than hard-coding them. Point-level window evaluation is not the official event-based NAB score.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

FILE = "/content/ambient_temperature_system_failure.csv"
if not os.path.exists(FILE):
    try:
        from google.colab import files
        uploaded = files.upload()
        FILE = next((n for n in uploaded if n.lower().endswith(".csv")), None)
    except ImportError:
        FILE = input("Path to ambient_temperature_system_failure.csv: ").strip()
if not FILE or not os.path.exists(FILE):
    import urllib.request
    FILE = "ambient_temperature_system_failure.csv"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/ambient_temperature_system_failure.csv",
        FILE
    )

df = pd.read_csv(FILE)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
if not {"timestamp", "value"}.issubset(df.columns):
    raise ValueError("CSV must contain timestamp and value columns.")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df = (df.dropna(subset=["timestamp", "value"]).sort_values("timestamp")
        .drop_duplicates("timestamp").reset_index(drop=True))

WINDOW = 50
history = df["value"].shift(1)
df["baseline"] = history.rolling(WINDOW, min_periods=WINDOW).median()
mad = history.rolling(WINDOW, min_periods=WINDOW).apply(
    lambda x: np.median(np.abs(x - np.median(x))), raw=True
)
std = history.rolling(WINDOW, min_periods=WINDOW).std(ddof=0)
scale = (1.4826 * mad).where(mad > 0, std).replace(0, np.nan)
df["score"] = (df["value"] - df["baseline"]).abs() / scale

windows = [
    ("2013-12-15 07:00:00", "2013-12-30 09:00:00"),
    ("2014-03-29 15:00:00", "2014-04-20 22:00:00"),
]
df["actual_anomaly"] = False
for start, end in windows:
    df.loc[df["timestamp"].between(start, end), "actual_anomaly"] = True

valid = df["score"].notna()
evaluated = df.loc[valid]
y_true = evaluated["actual_anomaly"].astype(int)

thresholds = [1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 3.0, 3.5]
rows = []
for t in thresholds:
    pred = (evaluated["score"] > t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "flagged": int(pred.sum())
    })
results = pd.DataFrame(rows).sort_values(
    ["F1", "precision", "threshold"], ascending=[False, False, True]
).reset_index(drop=True)
BEST_THRESHOLD = float(results.iloc[0]["threshold"])
display(results.round(3))
print(f"Rows: {len(df):,} | Evaluated: {int(valid.sum()):,} | Best threshold: {BEST_THRESHOLD}")

df["prediction"] = "NOT_EVALUATED"
df.loc[valid, "prediction"] = np.where(
    df.loc[valid, "score"] > BEST_THRESHOLD, "ANOMALOUS", "NORMAL"
)
df["explanation"] = np.select(
    [df["prediction"].eq("ANOMALOUS"), df["prediction"].eq("NORMAL")],
    ["Large deviation from recent baseline", "Within selected threshold"],
    default="Insufficient history or zero variation"
)
pred = (evaluated["score"] > BEST_THRESHOLD).astype(int)
print(df["prediction"].value_counts().to_string())
print(f"Precision: {precision_score(y_true, pred, zero_division=0):.3f} | "
      f"Recall: {recall_score(y_true, pred, zero_division=0):.3f} | "
      f"F1: {f1_score(y_true, pred, zero_division=0):.3f}")

anomalies = df[df["prediction"].eq("ANOMALOUS")]
plt.figure(figsize=(13, 5))
plt.plot(df["timestamp"], df["value"], label="Observed", linewidth=.7)
plt.plot(df["timestamp"], df["baseline"], label="Rolling baseline", linewidth=.8)
plt.scatter(anomalies["timestamp"], anomalies["value"], marker="x", s=18,
            label="Detected anomalies")
for i, (start, end) in enumerate(windows):
    plt.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=.12,
                label="Labelled anomaly window" if i == 0 else None)
plt.title(f"Sentinel — Best threshold {BEST_THRESHOLD:g}")
plt.xlabel("Timestamp"); plt.ylabel("Value")
plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

OUTPUT = "sentinel_final_predictions.csv"
df[["timestamp", "value", "baseline", "score", "prediction", "explanation"]].to_csv(OUTPUT, index=False)
print(f"Saved {OUTPUT}")
try:
    from google.colab import files
    files.download(OUTPUT)
except ImportError:
    print(os.path.abspath(OUTPUT))


   threshold  precision  recall     F1  flagged
0       1.00      0.109   0.471  0.178     3125
1       1.25      0.113   0.348  0.171     2231
2       1.50      0.108   0.234  0.148     1571
3       1.75      0.117   0.172  0.140     1065
4       2.00      0.107   0.106  0.107     720
5       2.25      0.127   0.087  0.103     496
6       2.50      0.149   0.069  0.094     335
7       3.00      0.205   0.045  0.074     161
8       3.50      0.259   0.029  0.052     81

Rows: 7,267 | Evaluated: 7,217 | Best threshold: 1.0
prediction
NORMAL           4092
ANOMALOUS        3125
NOT_EVALUATED      50
Precision: 0.109 | Recall: 0.471 | F1: 0.178
Saved sentinel_final_predictions.csv
